# 🚀 SAIL Digital Twin — Colab Launcher

**One-notebook workflow** that runs everything end-to-end:
1. Install dependencies
2. Upload master dataset
3. Train the model
4. Launch the Streamlit dashboard with a public URL

**If you prefer step-by-step control**, use `01_sail_digital_twin_development.ipynb` instead.

## Step 1 — Install Dependencies

In [ ]:
!pip install -q pandas numpy scipy scikit-learn xgboost joblib \
                matplotlib seaborn plotly openpyxl pyarrow streamlit pyngrok
!npm install -g localtunnel 2>/dev/null
print('✅ Dependencies installed')

## Step 2 — Upload Project & Dataset

Either:
- **Option A:** Upload `sail_digital_twin/` folder + `master_dataset_with_annual_summary.xlsx` via the Files panel (📁 icon)
- **Option B:** If you have a zip, run the cell below after uploading `sail_digital_twin.zip` to /content/

In [ ]:
# Optional — unzip project if you uploaded a zip
import os, zipfile
if os.path.exists('/content/sail_digital_twin.zip'):
    with zipfile.ZipFile('/content/sail_digital_twin.zip', 'r') as z:
        z.extractall('/content/')
    print('✅ Project unzipped')

# Verify project structure
project_root = '/content/sail_digital_twin'
assert os.path.exists(project_root), f'Missing {project_root} — please upload it'

# Move dataset into project root if it was uploaded to /content/
if os.path.exists('/content/master_dataset_with_annual_summary.xlsx') and \
   not os.path.exists(f'{project_root}/master_dataset_with_annual_summary.xlsx'):
    import shutil
    shutil.copy('/content/master_dataset_with_annual_summary.xlsx',
                f'{project_root}/master_dataset_with_annual_summary.xlsx')
    print('✅ Dataset copied into project root')

assert os.path.exists(f'{project_root}/master_dataset_with_annual_summary.xlsx'), \
    'Master dataset not found. Upload master_dataset_with_annual_summary.xlsx to /content/'
print(f'\n✅ Project ready at: {project_root}')

## Step 3 — Train the Model

Executes the full development notebook. Takes ~2-3 minutes depending on compute.

In [ ]:
%cd /content/sail_digital_twin/notebooks
!jupyter nbconvert --to notebook --execute 01_sail_digital_twin_development.ipynb \
    --output 01_sail_digital_twin_development.executed.ipynb \
    --ExecutePreprocessor.timeout=600
print('\n✅ Training complete — artifacts saved to /content/sail_digital_twin/models/')
%cd /content/sail_digital_twin
!ls -la models/ data/

## Step 4 — Launch Streamlit Dashboard

Starts the dashboard and generates a public URL via localtunnel.

In [ ]:
%cd /content/sail_digital_twin
# Kill any existing streamlit processes
!pkill -f streamlit 2>/dev/null; sleep 1

# Start Streamlit in background
!streamlit run streamlit_app/app.py --server.port 8501 --server.headless true &>/content/streamlit_log.txt &

# Wait for Streamlit to come up
import time
time.sleep(6)
print('✅ Streamlit running on port 8501')

In [ ]:
# Get the local IP (needed as the localtunnel password)
import urllib.request
try:
    ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip()
    print(f'🔑 Localtunnel password (public IP): {ip}')
except Exception:
    print('Run `!curl https://loca.lt/mytunnelpassword` in a separate cell')

# Launch localtunnel — the printed https:// URL is your dashboard
!npx --yes localtunnel --port 8501

### 📝 How to Open the Dashboard

1. The cell above prints a URL like `https://xxxxxx.loca.lt` — click it
2. A browser warning appears asking for a password — enter the IP printed above the URL
3. Dashboard loads

### 📝 Alternative — ngrok (if localtunnel fails)

```python
from pyngrok import ngrok
ngrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN")  # Free from https://ngrok.com
public_url = ngrok.connect(8501)
print(public_url)
```